# CUDA Demo Validation

Validasi startup corpus penuh, benchmark 60 kasus, dan flow demo Gradio pada GPU. Gunakan runtime baru, pilih **Runtime > Change runtime type > T4 GPU**, lalu jalankan **Run all**. Notebook akan mengunduh artefak `cuda_demo_validation.json` jika seluruh pemeriksaan berhasil.

In [ ]:
import subprocess

gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
).strip()
assert gpu, "Pilih Runtime > Change runtime type > T4 GPU"
print(gpu)

In [ ]:
import os
import subprocess
from pathlib import Path

BRANCH = "main"
repo_dir = Path("/content/indonesian-legal-compliance-rag")
if not (repo_dir / ".git").is_dir():
    subprocess.run(
        [
            "git", "clone", "--depth", "1", "--branch", BRANCH,
            "https://github.com/FadhilahAfif/indonesian-legal-compliance-rag.git",
            str(repo_dir),
        ],
        check=True,
    )
else:
    subprocess.run(["git", "checkout", BRANCH], cwd=repo_dir, check=True)
    subprocess.run(
        ["git", "pull", "--ff-only", "origin", BRANCH],
        cwd=repo_dir,
        check=True,
    )
os.chdir(repo_dir)
print(subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())

In [ ]:
import subprocess
import sys

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
    check=True,
)
subprocess.run(
    [
        sys.executable, "-m", "pip", "uninstall", "-y", "-q",
        "torchvision", "torchcodec", "torchaudio",
    ],
    check=True,
)
subprocess.run([sys.executable, "scripts/check_environment.py"], check=True)
subprocess.run(
    [sys.executable, "-m", "unittest", "discover", "-s", "tests", "-v"],
    check=True,
)

In [ ]:
import subprocess
import sys
from pathlib import Path
from src.rag import REGULATION_BY_FILE

subprocess.run(
    [
        sys.executable, "-m", "gdown", "--folder",
        "https://drive.google.com/drive/folders/1LHZ1IncPmmUN5kytFu3i7MoaafFrKDql",
        "-O", "data/raw",
    ],
    check=True,
)
corpus_dir = Path("data/raw")
missing = [name for name in REGULATION_BY_FILE if not (corpus_dir / name).is_file()]
assert not missing, f"PDF corpus tidak lengkap: {missing}"
print(f"Corpus OK: {len(REGULATION_BY_FILE)} PDF")

In [ ]:
import json
import subprocess
import sys
from pathlib import Path

subprocess.run(
    [sys.executable, "-m", "eval.validate_cases", "--require-reviewed"],
    check=True,
)
subprocess.run(
    [
        sys.executable, "-m", "eval.run_grounded",
        "--retrieval-predictions",
        "eval/results/final-retrieval/retrieval_predictions.jsonl",
        "--review", "eval/results/grounded-generation/grounded_review.json",
    ],
    check=True,
)
benchmark_path = Path("eval/results/grounded-generation/grounded_report.json")
benchmark = json.loads(benchmark_path.read_text(encoding="utf-8"))
assert benchmark["case_count"] == 60
assert benchmark["metrics"]["generation"]["faithfulness"] >= 0.90
assert benchmark["metrics"]["generation"]["citation_precision"] >= 0.90
assert benchmark["metrics"]["generation"]["valid_output_rate"] == 1.0
assert benchmark["metrics"]["safety"]["abstention_accuracy"] >= 0.85
print(json.dumps(benchmark["metrics"], indent=2))

In [ ]:
import json
import platform
import subprocess
from datetime import datetime, timezone

import torch

from app import NO_SOURCES, answer_question, build_app, build_search

assert torch.__version__.split("+", 1)[0] == "2.10.0", torch.__version__
assert torch.cuda.is_available(), "CUDA tidak tersedia setelah instalasi"
search = build_search(corpus_dir, "cuda")
demo = build_app(search)
assert demo.config["components"] and demo.config["dependencies"]

cases = [
    {
        "name": "answerable",
        "question": "Apa bentuk perizinan usaha berisiko menengah rendah?",
        "expected_status": "answer",
    },
    {
        "name": "unanswerable",
        "question": "Apa izin menurut PP Nomor 28 Tahun 2025?",
        "expected_status": "insufficient_context",
    },
]
results = []
for case in cases:
    messages, status, elapsed, sources, debug = answer_question(
        case["question"], [], search
    )
    assert debug.get("final_status") == case["expected_status"], debug
    if case["expected_status"] == "answer":
        assert debug["retrieved"] and sources != NO_SOURCES
    else:
        assert not debug["retrieved"]
    results.append(
        {
            **case,
            "status_markdown": status,
            "elapsed_markdown": elapsed,
            "answer_markdown": messages[-1]["content"],
            "source_cards_markdown": sources,
            "debug": debug,
        }
    )
    print(f"{case['name']}: {debug['final_status']} ({elapsed})")

artifact = {
    "passed": True,
    "validated_at_utc": datetime.now(timezone.utc).isoformat(),
    "git_commit": subprocess.check_output(
        ["git", "rev-parse", "HEAD"], text=True
    ).strip(),
    "environment": {
        "python": platform.python_version(),
        "torch": torch.__version__,
        "cuda": torch.version.cuda,
        "gpu": torch.cuda.get_device_name(0),
    },
    "corpus_files": sorted(REGULATION_BY_FILE),
    "ui": {
        "components": len(demo.config["components"]),
        "dependencies": len(demo.config["dependencies"]),
    },
    "benchmark": {
        "case_count": benchmark["case_count"],
        "metrics": benchmark["metrics"],
    },
    "results": results,
}
output = Path("eval/results/demo-validation/cuda_demo_validation.json")
output.parent.mkdir(parents=True, exist_ok=True)
output.write_text(
    json.dumps(artifact, ensure_ascii=False, indent=2) + "\n", encoding="utf-8"
)
print(json.dumps({"passed": True, "artifact": str(output)}, indent=2))

In [ ]:
from google.colab import files

files.download(str(output))